今日课程主线：为什么语言需要专门建模 → 文本预处理 → 中文分词 → 子词 Tokenizer → 数字编码 → 词向量 → Word2Vec。

**今天主要解决的问题：怎样把一段变长、有顺序、依赖上下文的文字，变成神经网络能使用的向量？**

```text
语言：变长 + 有序 + 元素相关 + 离散符号
    ↓
先决定按什么单位处理文本
    ├─ 中文词级分词：词典 / HMM → 词
    └─ 子词 Tokenizer：BPE / WordPiece / Unigram → token
    ↓
建立词表：token ↔ ID
    ↓
Embedding：ID → 连续向量
    ↓
Word2Vec：借上下文预测任务训练这张向量表
    ├─ CBOW：上下文 → 中心词
    └─ Skip-gram：中心词 → 上下文
    ↓
得到能表达部分语义关系的静态向量
    ↓
后续：序列模型 / Attention / Transformer → 上下文化表示
```

**HMM 是一种分词路线，不是 BPE 的前置步骤；Word2Vec 是学习词向量的一种方法，也不是所有大模型都必须先执行的步骤。** 现代语言模型通常直接在自身的训练目标下学习 token embedding。

| 标签 | 含义 | 例子 |
|---|---|---|
| B：Begin | 多字词的词首 | “鱼香肉丝”的“鱼” |
| M：Middle | 多字词的中间 | “香”“肉” |
| E：End | 多字词的词尾 | “丝” |
| S：Single | 单字成词 | 单独成词的“的” |

```text
单字词：S
双字词：B E
三字词：B M E
四字词：B M M E
```


In [3]:
# jieba 分词示例
import jieba

text = "我来到成都大学，学习人工智能和计算机视觉。"

# 1. 精确模式（默认）：适合文本分析
seg = jieba.lcut(text)
print("精确模式:", seg)

# 2. 全模式：把所有可能成词的词语都扫描出来
seg_all = jieba.lcut(text, cut_all=True)
print("全模式  :", seg_all)

# 3. 搜索引擎模式：在精确模式基础上对长词再次切分
seg_search = jieba.lcut_for_search(text)
print("搜索模式:", seg_search)

# 4. 添加自定义词典
jieba.add_word("成都大学")
seg_custom = jieba.lcut(text)
print("自定义词:", seg_custom)

# 5. 关键词提取（需要先安装 jieba.analyse，自带）
import jieba.analyse
keywords = jieba.analyse.extract_tags(text, topK=5)
print("关键词  :", keywords)

精确模式: ['我', '来到', '成都大学', '，', '学习', '人工智能', '和', '计算机', '视觉', '。']
全模式  : ['我', '来到', '成都', '成都大学', '大学', '，', '学习', '人工', '人工智能', '智能', '和', '计算', '计算机', '算机', '视觉', '。']
搜索模式: ['我', '来到', '成都', '大学', '成都大学', '，', '学习', '人工', '智能', '人工智能', '和', '计算', '算机', '计算机', '视觉', '。']
自定义词: ['我', '来到', '成都大学', '，', '学习', '人工智能', '和', '计算机', '视觉', '。']
关键词  : ['成都大学', '人工智能', '视觉', '计算机', '学习']


In [8]:
import jieba
from collections import Counter

# ===== 语料 =====
sentence1 = "我在成都大学欢迎2026级新同学"
sentence2 = "我来到成都大学，学习人工智能和计算机视觉。"
corpus = [sentence1, sentence2]

# ===== 1. 分词（并过滤掉标点/空白等非词符号） =====
PUNCT = set("，。、！？；：…—·《》【】（）“”‘’'\n\t ")

def clean_words(text):
    """分词，仅保留有实际意义的词（去掉标点、空白）"""
    return [w for w in jieba.lcut(text) if w.strip() and not all(ch in PUNCT for ch in w)]

tokenized = [clean_words(s) for s in corpus]
print("分词结果:", tokenized)

# ===== 2. 展平所有词，统计词频 =====
all_words = [w for sent in tokenized for w in sent]
freq = Counter(all_words)
print("词频统计:", dict(freq))

# ===== 3. 构建词表 vocab（word <-> id 双向映射） =====
# NLP 中词表通常带特殊 token：<pad> 占位、<unk> 表示词表外的词
PAD, UNK = "<pad>", "<unk>"
vocab_words = [PAD, UNK] + [w for w, _ in freq.most_common()]
word2idx = {w: i for i, w in enumerate(vocab_words)}   # 词 -> id（正向）
idx2word = {i: w for w, i in word2idx.items()}          # id -> 词（反向）

print(f"\n词表大小: {len(word2idx)}（含特殊 token {PAD}、{UNK}）")
print("word2idx:", word2idx)
print("idx2word:", idx2word)

# ===== 4. 双向转换：句子 <-> id 序列 =====
def encode(text):
    """句子 -> id 序列（词表外词用 <unk> 代替）"""
    return [word2idx.get(w, word2idx[UNK]) for w in clean_words(text)]

def decode(ids):
    """id 序列 -> 词序列（反向查 idx2word，未知 id 显示 <unk>）"""
    return [idx2word.get(i, UNK) for i in ids]

print()
for s in corpus:
    ids = encode(s)
    print(f"句子 -> ids : {s}  ->  {ids}")
    print(f"ids  -> 词 : {ids}  ->  {decode(ids)}")

分词结果: [['我', '在', '成都', '大学', '欢迎', '2026', '级', '新', '同学'], ['我', '来到', '成都', '大学', '学习', '人工智能', '和', '计算机', '视觉']]
词频统计: {'我': 2, '在': 1, '成都': 2, '大学': 2, '欢迎': 1, '2026': 1, '级': 1, '新': 1, '同学': 1, '来到': 1, '学习': 1, '人工智能': 1, '和': 1, '计算机': 1, '视觉': 1}

词表大小: 17（含特殊 token <pad>、<unk>）
word2idx: {'<pad>': 0, '<unk>': 1, '我': 2, '成都': 3, '大学': 4, '在': 5, '欢迎': 6, '2026': 7, '级': 8, '新': 9, '同学': 10, '来到': 11, '学习': 12, '人工智能': 13, '和': 14, '计算机': 15, '视觉': 16}
idx2word: {0: '<pad>', 1: '<unk>', 2: '我', 3: '成都', 4: '大学', 5: '在', 6: '欢迎', 7: '2026', 8: '级', 9: '新', 10: '同学', 11: '来到', 12: '学习', 13: '人工智能', 14: '和', 15: '计算机', 16: '视觉'}

句子 -> ids : 我在成都大学欢迎2026级新同学  ->  [2, 5, 3, 4, 6, 7, 8, 9, 10]
ids  -> 词 : [2, 5, 3, 4, 6, 7, 8, 9, 10]  ->  ['我', '在', '成都', '大学', '欢迎', '2026', '级', '新', '同学']
句子 -> ids : 我来到成都大学，学习人工智能和计算机视觉。  ->  [2, 11, 3, 4, 12, 13, 14, 15, 16]
ids  -> 词 : [2, 11, 3, 4, 12, 13, 14, 15, 16]  ->  ['我', '来到', '成都', '大学', '学习', '人工智能', '和', '计算机', '视觉']


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1. 构造简单语料
sentence = [
    "我",
    "爱",
    "自然",
    "语言",
    "处理"
]

# 词表
word2idx = {
    word: idx
    for idx, word in enumerate(sentence)
}

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

vocab_size = len(word2idx)
# embedding维度
embedding_dim = 3

# 2. 构造CBOW训练样本
samples = []

for i in range(1, len(sentence)-1):
    context = [
        sentence[i-1],
        sentence[i+1]
    ]
    target = sentence[i]
    samples.append(
        (context, target)
    )
print("训练样本:")
for s in samples:
    print(s)

# 3. one-hot编码
def one_hot(index, size):

    vec = torch.zeros(size)

    vec[index] = 1

    return vec
# 4. CBOW模型

class CBOW(nn.Module):
    def __init__(self,vocab_size,embedding_dim):
        super().__init__()
        # 第一层：
        # one-hot -> word vector
        self.embedding = nn.Linear(
            vocab_size,
            embedding_dim,
            bias=False
        )
        # 输出层：
        # embedding -> vocab概率
        self.output = nn.Linear(
            embedding_dim,
            vocab_size
        )
    def forward(self, context):
        # 每个词映射成embedding
        emb = self.embedding(context)
        # CBOW:
        # 多个上下文向量求平均
        hidden = emb.mean(dim=0)
        # 分类预测中心词
        logits = self.output(hidden)
        return logits

# 5. 创建模型

model = CBOW(vocab_size, embedding_dim)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.1
)
# 6. 开始训练

epochs = 500
for epoch in range(epochs):
    total_loss = 0
    for context_words, target_word in samples:
        # 上下文one-hot
        context_vectors = torch.stack(
            [
                one_hot(
                    word2idx[w],
                    vocab_size
                )
                for w in context_words
            ]
        )
        # 标签
        target = torch.tensor(
            word2idx[target_word],
            dtype=torch.long
        )
        # 清梯度
        optimizer.zero_grad()
        # forward
        logits = model(
            context_vectors
        )
        # loss
        loss = criterion(
            logits.unsqueeze(0),
            target.unsqueeze(0)
        )
        # backward
        loss.backward()
        # 更新参数
        optimizer.step()
        total_loss += loss.item()

    if epoch % 50 == 0:
        print(
            f"epoch={epoch}, loss={total_loss:.4f}"
        )

# 7. 查看训练后的词向量

print("\n训练后的词向量:")

weight = model.embedding.weight.detach()
for word in sentence:
    idx = word2idx[word]
    # Linear权重:
    # shape=[embedding_dim,vocab_size]
    vector = weight[:, idx]
    print(
        word,
        vector.numpy()
    )

# 8. 计算词向量余弦相似度

def get_vector(word):
    idx = word2idx[word]
    return weight[:, idx]

v1 = get_vector("自然")
v2 = get_vector("语言")

similarity = F.cosine_similarity(
    v1.unsqueeze(0),
    v2.unsqueeze(0)
)

print(
    "\n自然 与 语言 相似度:",
    similarity.item()
)


训练样本:
(['我', '自然'], '爱')
(['爱', '语言'], '自然')
(['自然', '处理'], '语言')
epoch=0, loss=5.3066
epoch=50, loss=0.3420
epoch=100, loss=0.0818
epoch=150, loss=0.0416
epoch=200, loss=0.0269
epoch=250, loss=0.0196
epoch=300, loss=0.0152
epoch=350, loss=0.0124
epoch=400, loss=0.0104
epoch=450, loss=0.0089

训练后的词向量:
我 [-1.1891329   0.41398713  2.3659585 ]
爱 [1.208989   1.1869842  0.49413118]
自然 [-1.3341268  -0.91967523  0.59140193]
语言 [ 1.5961245   1.2082769  -0.14307573]
处理 [ 0.07190705 -2.1576333  -1.884411  ]

自然 与 语言 相似度: -0.9605162739753723
